# pile-scan 1: scan a pile of pages

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/daverooney/pile-scan/blob/main/notebooks/01_scan.ipynb)

Reads every image and PDF in a Google Drive folder with **Gemma 4 12B** running on this notebook's GPU, and writes a transcription plus ordering clues for each page to a SQLite database on Drive. Nothing leaves the notebook: no hosted model APIs.

**It's safe to kill.** The database is copied to Drive every few pages. When Colab disconnects, reconnect and **Run all** again: it restores the database and skips every page already done.

1. **Runtime > Change runtime type**, pick a GPU (T4 on the free tier, L4 on Pro).
2. Fill in the settings below.
3. **Runtime > Run all**.

In [ ]:
#@title Settings { display-mode: "form" }
#@markdown **Folders on your Drive** (as seen after mounting)
INPUT_DIR = "/content/drive/MyDrive/pile-scan/inbox"  #@param {type:"string"}
OUTPUT_DIR = "/content/drive/MyDrive/pile-scan/output"  #@param {type:"string"}
DB_NAME = "pile_scan.db"  #@param {type:"string"}
RECURSIVE = True  #@param {type:"boolean"}

#@markdown **Model.** 4bit fits a free T4 (16 GB). 8bit needs about 14 GB for weights alone, so use an L4 (24 GB) for it.
MODEL_ID = "google/gemma-4-12B-it"  #@param {type:"string"}
QUANTIZATION = "4bit"  #@param ["4bit", "8bit", "none"]
#@markdown Visual tokens per page. 1120 keeps the most detail for small print; Transformers' default is 280.
MAX_SOFT_TOKENS = 1120  #@param [70, 140, 280, 560, 1120] {type:"raw"}
MAX_NEW_TOKENS = 3072  #@param {type:"integer"}

#@markdown **Run control**
CHECKPOINT_EVERY = 10  #@param {type:"integer"}
#@markdown Stop after this many pages (0 = no limit). Same number gives the same pages every time.
SAMPLE_LIMIT = 0  #@param {type:"integer"}
RETRY_ERRORS = False  #@param {type:"boolean"}
PDF_DPI = 200  #@param {type:"integer"}
#@markdown Release the GPU when the scan ends, so a finished run stops using compute units.
DISCONNECT_WHEN_DONE = False  #@param {type:"boolean"}

#@markdown **Code to install** (the branch is useful for trying changes before they merge)
REPO = "daverooney/pile-scan"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}

# Factual questions asked of every page. Keys become JSON keys in `answers_json`.
QUESTIONS = {
    "summary": "One sentence describing what this page is about.",
    "author": "Who wrote or sent this document, if shown? Otherwise null.",
    "recipient": "Who is this document addressed to, if shown? Otherwise null.",
    "organizations": "List the organizations named on the page (array of strings).",
}

In [ ]:
#@title Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
#@title Install
# If the repo is private, add a Colab secret named GITHUB_TOKEN (key icon on the left)
# holding a read-only GitHub token, and give this notebook access to it.
import subprocess, sys

def _secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None

_token = _secret("GITHUB_TOKEN")
_auth = f"{_token}@" if _token else ""
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "transformers>=5.15", "accelerate>=1.0", "bitsandbytes>=0.45", "pypdfium2>=4.30"], check=True)
# --force-reinstall so re-running this cell picks up new commits on BRANCH.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "--no-deps",
                f"git+https://{_auth}github.com/{REPO}.git@{BRANCH}"], check=True)

# Gemma 4 is Apache-2.0 licensed. If Hugging Face ever asks for a login, add an
# HF_TOKEN secret and it will be used here.
_hf = _secret("HF_TOKEN")
if _hf:
    from huggingface_hub import login
    login(token=_hf)

import pile_scan
print("pile-scan", pile_scan.__version__)

In [ ]:
#@title Check the pile (no GPU work)
from pile_scan import ScanConfig
from pile_scan.scanner import run_scan

cfg = ScanConfig(
    input_dir=INPUT_DIR, output_dir=OUTPUT_DIR, db_name=DB_NAME, recursive=RECURSIVE,
    model_id=MODEL_ID, quantization=QUANTIZATION, max_soft_tokens=int(MAX_SOFT_TOKENS),
    max_new_tokens=MAX_NEW_TOKENS, checkpoint_every=CHECKPOINT_EVERY,
    sample_limit=SAMPLE_LIMIT or None, retry_errors=RETRY_ERRORS, pdf_dpi=PDF_DPI,
    questions=QUESTIONS,
)
cfg.validate()
# Hashes every file and counts pages. Hashes are cached, so later runs only
# hash new or changed files.
inventory = run_scan(cfg, None)
print(f"{inventory.pages_total_known} pages in {inventory.files_seen} files; "
      f"{inventory.pages_skipped} already done.")

In [ ]:
#@title Load the model
import torch
from pile_scan.model import GemmaPageReader, gpu_name

gpu = gpu_name()
if gpu is None:
    raise SystemExit("No GPU. Runtime > Change runtime type > pick a GPU, then Run all again.")
total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {gpu} ({total_gb:.0f} GB)")
if QUANTIZATION != "4bit" and total_gb < 20:
    print(f"Warning: {QUANTIZATION} may not fit in {total_gb:.0f} GB. Use 4bit on a T4.")

reader = GemmaPageReader(cfg)
print(f"Loaded {MODEL_ID} ({QUANTIZATION}); GPU memory in use: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

In [ ]:
#@title Scan
# Safe to stop at any time (Runtime > Interrupt execution): progress is saved.
summary = run_scan(cfg, reader, gpu_name=gpu)

if DISCONNECT_WHEN_DONE and summary.end_reason != "interrupted":
    drive.flush_and_unmount()   # make sure the last checkpoint reaches Drive
    from google.colab import runtime
    runtime.unassign()

## Look at the results

The database on Drive is a normal SQLite file. Main tables:

- `pages`: one row per page with `ocr_text`, the ordering clues (`doc_title`, `page_number_printed`, `primary_date`, `first_line`, `last_line`, `running_header`, `ends_mid_sentence`, ...) and `answers_json`.
- `files`: one row per distinct file, keyed by its SHA-256; `file_paths` lists every path it was found at.
- `runs`: one row per scan run with its settings.

In [ ]:
#@title Recent pages
import sqlite3, pandas as pd
con = sqlite3.connect(str(cfg.local_db_path))
pd.read_sql("""
    SELECT f.source_path, p.page_index + 1 AS page, p.status, p.doc_title, p.doc_type,
           p.page_number_printed, p.primary_date, p.truncated, ROUND(p.elapsed_s, 1) AS secs,
           substr(p.ocr_text, 1, 120) AS text_start
    FROM pages p JOIN files f USING (file_sha256)
    ORDER BY p.created_at DESC LIMIT 25
""", con)

## Optional: compare 4-bit with 8-bit

1. Set `SAMPLE_LIMIT` to 20, `QUANTIZATION` to `4bit`, `DB_NAME` to `sample_4bit.db`, and Run all.
2. Switch to an L4 runtime, set `QUANTIZATION` to `8bit`, `DB_NAME` to `sample_8bit.db`, and Run all.
3. Run the cell below. Both runs read the same 20 pages, because files are visited in sorted order.

Numbers (dates, amounts, IDs) are where lower precision tends to slip first, so look hardest at `numbers_match`.

In [ ]:
#@title Compare two sample runs
DB_A = "/content/drive/MyDrive/pile-scan/output/sample_4bit.db"  #@param {type:"string"}
DB_B = "/content/drive/MyDrive/pile-scan/output/sample_8bit.db"  #@param {type:"string"}
import pandas as pd
from pile_scan import compare
rows = compare.compare(DB_A, DB_B)
print(compare.summarize(rows))
pd.DataFrame(rows).sort_values("text_similarity")